# Lesson 5.5 — Human Review Workflows & Confidence Calibration

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why aggregate accuracy can mask poor performance on specific types/fields
- Use stratified random sampling to measure error rates and detect novel patterns
- Calibrate field-level confidence using labeled validation sets
- Route low-confidence or ambiguous extractions to human review

**You will produce:** A stratified sampler, a per-segment accuracy report, and a confidence-based review router.

## Concepts

**Aggregate accuracy hides problems.** A 97% overall rate can mask poor performance on a specific **document type** or **field**. Before automating high-confidence extractions, validate accuracy **by document type and field segment**.

- **Stratified random sampling** of high-confidence extractions measures ongoing error rates and surfaces **novel** error patterns.
- **Field-level confidence scores**, **calibrated with labeled validation sets**, route review attention where it matters.
- **Route to human review** extractions with low model confidence or ambiguous/contradictory source documents, to prioritize limited reviewer capacity.

Note the tie-back to Lesson 5.2: self-reported confidence is weak for **escalation** decisions, but **calibrated** field-level confidence (validated against labels) is useful for **review routing**.

> **Reference:** [Anthropic - evaluate & strengthen guardrails](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Stratified sampling by document type

Sample within each stratum so rare types aren't drowned out by common ones.

In [ ]:
import random
random.seed(0)
def stratified_sample(records, key, frac=0.2):
    strata = {}
    for r in records:
        strata.setdefault(r[key], []).append(r)
    sample = []
    for group in strata.values():
        k = max(1, int(len(group) * frac))
        sample += random.sample(group, k)
    return sample

recs = [{"id": i, "doc_type": ("invoice" if i % 5 else "receipt")} for i in range(50)]
s = stratified_sample(recs, "doc_type")
from collections import Counter
print("sampled by type:", Counter(r["doc_type"] for r in s))

## Code-Along: Step 2 — Per-segment accuracy report

Break accuracy down by type and field to expose hidden weak spots.

In [ ]:
def segment_accuracy(results):
    agg = {}
    for r in results:
        seg = (r["doc_type"], r["field"])
        a = agg.setdefault(seg, [0, 0])
        a[1] += 1
        a[0] += 1 if r["correct"] else 0
    return {seg: round(c / n, 2) for seg, (c, n) in agg.items()}

data = [{"doc_type": "receipt", "field": "total", "correct": False},
        {"doc_type": "receipt", "field": "total", "correct": True},
        {"doc_type": "invoice", "field": "total", "correct": True}]
print(segment_accuracy(data))  # receipt/total weak despite high overall

## Code-Along: Step 3 — Confidence-based review routing

Low confidence or conflicting sources go to a human.

In [ ]:
def route(extraction, threshold=0.7):
    if extraction.get("conflict_detected") or extraction.get("confidence", 1.0) < threshold:
        return "human_review"
    return "auto_accept"

print(route({"confidence": 0.55}))
print(route({"confidence": 0.95, "conflict_detected": True}))
print(route({"confidence": 0.9}))

## Exercise

Write `calibrate(threshold, labeled_set)` that reports the human-review rate and the missed-error rate at a given confidence threshold, using a labeled validation set. Find a threshold where receipt/total errors are caught. Assert that lowering the threshold catches more errors but routes more items to humans (the calibration tradeoff).

In [ ]:
# TODO: implement calibrate(threshold, labeled_set) -> {review_rate, missed_error_rate}
# TODO: assert lower threshold -> higher review_rate, lower missed_error_rate

## Key Takeaways

- Aggregate accuracy can hide poor performance on specific document types/fields.
- Use stratified random sampling to measure error rates and find novel patterns.
- Calibrate field-level confidence with labeled validation sets for review routing.
- Route low-confidence or contradictory extractions to human review.

---

**Next:** [Lesson 5.6 — Provenance & Uncertainty in Synthesis](./lesson-5.6-provenance-synthesis.ipynb)